# 0 - Imports & Settings

In [ ]:
import os
import json
import time
import zipfile
import datetime
import warnings
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
from tqdm import tqdm
from collections import defaultdict

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1 - Helper Classes for
- Utitlity Functions
- Data Loading
- Visualizations
- Scoring Functions

### Utility

In [ ]:
class Utility:
    @staticmethod
    def add_noise(data, mask, ng, pixel_size=2.):
        """
        Add noise to a noiseless convergence map.

        Parameters
        ----------
        data : np.array
            Noiseless convergence maps.
        mask : np.array
            Binary mask map.
        ng : float
            Number of galaxies per arcmin². This determines the noise level; a larger number means smaller noise.
        pixel_size : float, optional
            Pixel size in arcminutes (default is 2.0).
        """

        return data + np.random.randn(*data.shape) * 0.4 / (2*ng*pixel_size**2)**0.5 * mask

    @staticmethod
    def load_np(data_dir, file_name):
        file_path = os.path.join(data_dir, file_name)
        return np.load(file_path, mmap_mode='r')

    @staticmethod
    def save_np(data_dir, file_name, data):
        file_path = os.path.join(data_dir, file_name)
        np.save(file_path, data)

    @staticmethod
    def save_json_zip(submission_dir, json_file_name, zip_file_name, data):
        """
        Save a dictionary with 'means' and 'errorbars' into a JSON file,
        then compress it into a ZIP file inside submission_dir.

        Parameters
        ----------
        submission_dir : str
            Path to the directory where the ZIP file will be saved.
        file_name : str
            Name of the ZIP file (without extension).
        data : dict
            Dictionary with keys 'means' and 'errorbars'.

        Returns
        -------
        str
            Path to the created ZIP file.
        """
        os.makedirs(submission_dir, exist_ok=True)

        json_path = os.path.join(submission_dir, json_file_name)

        # Save JSON file
        with open(json_path, "w") as f:
            json.dump(data, f)

        # Path to ZIP
        zip_path = os.path.join(submission_dir, zip_file_name)

        # Create ZIP containing only the JSON
        with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
            zf.write(json_path, arcname=json_file_name)

        # Remove the standalone JSON after zipping
        os.remove(json_path)

        return zip_path

### Data

In [ ]:
class Data:
    def __init__(self, data_dir, USE_PUBLIC_DATASET):
        self.USE_PUBLIC_DATASET = USE_PUBLIC_DATASET
        self.data_dir = data_dir
        self.mask_file = 'WIDE12H_bin2_2arcmin_mask.npy'
        self.viz_label_file = 'label.npy'
        if self.USE_PUBLIC_DATASET:
            self.kappa_file = 'WIDE12H_bin2_2arcmin_kappa.npy'
            self.label_file = self.viz_label_file
            self.Ncosmo = 101  # Number of cosmologies in the entire training data
            self.Nsys = 256    # Number of systematic realizations in the entire training data
            self.test_kappa_file = 'WIDE12H_bin2_2arcmin_kappa_noisy_test.npy'
            self.Ntest = 4000  # Number of instances in the test data




        else:
            self.kappa_file = 'sampled_WIDE12H_bin2_2arcmin_kappa.npy'
            self.label_file = 'sampled_label.npy'
            self.Ncosmo = 3    # Number of cosmologies in the sampled training data
            self.Nsys = 30     # Number of systematic realizations in the sampled training data
            self.test_kappa_file = 'sampled_WIDE12H_bin2_2arcmin_kappa_noisy_test.npy'
            self.Ntest = 3     # Number of instances in the sampled test data

        self.shape = [1424,176] # dimensions of each map
        self.pixelsize_arcmin = 2 # pixel size in arcmin
        self.pixelsize_radian = self.pixelsize_arcmin / 60 / 180 * np.pi # pixel size in radian
        self.ng = 30  # galaxy number density. This determines the noise level of the experiment. Do not change this number.

    def load_train_data(self):
        self.mask = Utility.load_np(data_dir=self.data_dir, file_name=self.mask_file) # A binary map that shows which parts of the sky are observed and which areas are blocked
        self.kappa = np.zeros((self.Ncosmo, self.Nsys, *self.shape), dtype=np.float16)
        self.kappa[:,:,self.mask] = Utility.load_np(data_dir=self.data_dir, file_name=self.kappa_file) # Training convergence maps
        self.label = Utility.load_np(data_dir=self.data_dir, file_name=self.label_file) # Training labels (cosmological and physical paramameters) of each training map
        self.viz_label = Utility.load_np(data_dir=self.data_dir, file_name=self.viz_label_file) # For visualization of parameter distributions

    def load_test_data(self):
        self.kappa_test = np.zeros((self.Ntest, *self.shape), dtype=np.float16)
        self.kappa_test[:,self.mask] = Utility.load_np(data_dir=self.data_dir, file_name=self.test_kappa_file) # Test noisy convergence maps

In [ ]:
class CosmologyDataset(Dataset):
    """
    Custom PyTorch Dataset
    """

    def __init__(self, data, labels=None, transform=None, label_transform=None):
        self.data = data
        self.labels = labels
        self.transform = transform
        self.label_transform = label_transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        image = self.data[idx].astype(np.float32)   # Convert from float16 to float32
        if self.transform:
            image = self.transform(image)
        if self.labels is not None:
            label = self.labels[idx].astype(np.float32)
            label = torch.from_numpy(label)
            if self.label_transform:
                label = self.label_transform(label)
            return image, label
        else:
            return image

### Visualization

In [ ]:
class Visualization:

    @staticmethod
    def plot_mask(mask):
        plt.figure(figsize=(30,100))
        plt.imshow(mask.T)
        plt.show()

    @staticmethod
    def plot_noiseless_training_convergence_map(kappa):
        plt.figure(figsize=(30,100))
        plt.imshow(kappa[0,0].T, vmin=-0.02, vmax=0.07)
        plt.show()

    @staticmethod
    def plot_noisy_training_convergence_map(kappa, mask, pixelsize_arcmin, ng):
        plt.figure(figsize=(30,100))
        plt.imshow(Utility.add_noise(kappa[0,0], mask, ng, pixelsize_arcmin).T, vmin=-0.02, vmax=0.07)
        plt.show()

    @staticmethod
    def plot_cosmological_parameters_OmegaM_S8(label):
        plt.scatter(label[:,0,0], label[:,0,1])
        plt.xlabel(r'$\Omega_m$')
        plt.ylabel(r'$S_8$')
        plt.show()

    @staticmethod
    def plot_baryonic_physics_parameters(label):
        plt.scatter(label[0,:,2], label[0,:,3])
        plt.xlabel(r'$T_{\mathrm{AGN}}$')
        plt.ylabel(r'$f_0$')
        plt.show()

    @staticmethod
    def plot_photometric_redshift_uncertainty_parameters(label):
        plt.hist(label[0,:,4], bins=20)
        plt.xlabel(r'$\Delta z$')
        plt.show()

### Scoring function

In [ ]:
class Score:
    @staticmethod
    def _score_phase1(true_cosmo, infer_cosmo, errorbar):
        """
        Computes the log-likelihood score for Phase 1 based on predicted cosmological parameters.

        Parameters
        ----------
        true_cosmo : np.ndarray
            Array of true cosmological parameters (shape: [n_samples, n_params]).
        infer_cosmo : np.ndarray
            Array of inferred cosmological parameters from the model (same shape as true_cosmo).
        errorbar : np.ndarray
            Array of standard deviations (uncertainties) for each inferred parameter
            (same shape as true_cosmo).

        Returns
        -------
        np.ndarray
            Array of scores for each sample (shape: [n_samples]).
        """

        sq_error = (true_cosmo - infer_cosmo)**2
        scale_factor = 1000  # This is a constant that scales the error term.
        score = - np.sum(sq_error / errorbar**2 + 2*np.log(errorbar) + scale_factor * sq_error, 1)
        score = np.mean(score)
        if score >= -10**6: # Set a minimum of the score (to properly display on Codabench)
            return score
        else:
            return -10**6

# 2 - Load train and test data

Shape of the training maps kappa: $(N_{\rm cosmo}, N_{\rm sys}, 1424, 176)$.

Shape of the labels: $(N_{\rm cosmo}, N_{\rm sys}, 5)$.

In [ ]:
root_dir = os.getcwd()
print("Root directory is", root_dir)

* `USE_PUBLIC_DATASET = False` - only downsampled data, $N_{\rm cosmo}=3$, $N_{\rm sys}=30$ for training and $N_{\rm test}=3$ for test.

* `USE_PUBLIC_DATASET = True` - entire data, $N_{\rm cosmo}=101$, $N_{\rm sys}=256$ for training and $N_{\rm test}=4000$ for test.

## Using huge dataset

In [ ]:
USE_PUBLIC_DATASET = True
PUBLIC_DATA_DIR = os.path.join(root_dir, 'drive/MyDrive/public_data/')
  # This is only required when you set USE_PUBLIC_DATASET = True

In [ ]:
if USE_PUBLIC_DATASET:
  from google.colab import drive
  drive.mount('/content/drive')
  %cd drive/MyDrive/

else:
  !git clone --depth 1 https://github.com/FAIR-Universe/Cosmology_Challenge.git


In [ ]:
print("Fichiers :", os.listdir())

In [ ]:
if not USE_PUBLIC_DATASET:                                         # Testing this startking kit with a tiny sample of the training data (3, 30, 1424, 176)
    DATA_DIR = os.path.join(root_dir, 'Cosmology_Challenge/input_data/')
else:                                                                # Training your model with all training data (101, 256, 1424, 176)
    DATA_DIR = PUBLIC_DATA_DIR

### Load the train and test data:

In [ ]:
# Initialize Data class object
data_obj = Data(data_dir=DATA_DIR, USE_PUBLIC_DATASET=USE_PUBLIC_DATASET)


In [ ]:
data_obj.load_train_data()

In [ ]:
# data_obj.load_test_data()

In [ ]:
Ncosmo = data_obj.Ncosmo
Nsys = data_obj.Nsys

print(f'There are {Ncosmo} cosmological models, each has {Nsys} realizations of nuisance parameters in the training data.')

In [ ]:
print(f'Shape of the training data = {data_obj.kappa.shape}')
print(f'Shape of the mask = {data_obj.mask.shape}')
print(f'Shape of the training label = {data_obj.label.shape}')
# print(f'Shape of the test data = {data_obj.kappa_test.shape}')

### Add noise:
- The original training images are *noiseless* (without any pixel-level noise).
- The original test images is *noisy* (pixel-level noise with galaxy number density $n_g = 30~\text{arcmin}^{-2}$ and pixel size $=2$ arcmin has been added).

In [ ]:
# Add the pixel-level noise to the training set (note that this may take some time and large memory)

np.random.seed(31415)  # Fix the random seed for reproducible results

noisy_kappa = Utility.add_noise(data=data_obj.kappa.astype(np.float64),
                                mask=data_obj.mask,
                                ng=data_obj.ng,
                                pixel_size=data_obj.pixelsize_arcmin)


In [ ]:
%cd ../..

In [ ]:
# del data_obj
# import gc
# gc.collect()

### Split training set:

If you want to split your own training/validation sets to evaluate your model, we recommend splitting the original training data along `axis = 1` (the 256 realizations of nuisance parameters). This will ensure that there are no intrinsic correlations between the training and validation sets.

In [ ]:
# Split the data into training and validation sets

NP_idx = np.arange(Nsys)  # The indices of Nsys nuisance parameter realizations
split_fraction = 0.2      # Set the fraction of data you want to split (between 0 and 1)
seed = 5566

train_NP_idx, val_NP_idx = train_test_split(NP_idx, test_size=split_fraction, random_state=seed)

noisy_kappa_train = noisy_kappa[:, train_NP_idx]      # shape = (Ncosmo, len(train_NP_idx), 1424, 176)
label_train = data_obj.label[:, train_NP_idx]         # shape = (Ncosmo, len(train_NP_idx), 5)
noisy_kappa_val = noisy_kappa[:, val_NP_idx]          # shape = (Ncosmo, len(val_NP_idx), 1424, 176)
label_val = data_obj.label[:, val_NP_idx]             # shape = (Ncosmo, len(val_NP_idx), 5)

Ntrain = label_train.shape[0]*label_train.shape[1]
Nval = label_val.shape[0]*label_val.shape[1]

In [ ]:
print(f'Shape of the split training data = {noisy_kappa_train.shape}')
print(f'Shape of the split validation data = {noisy_kappa_val.shape}')

print(f'Shape of the split training labels = {label_train.shape}')
print(f'Shape of the split validation labels = {label_val.shape}')

In [ ]:
# # Save the split data and labels for future usage

# Utility.save_np(data_dir=DATA_DIR, file_name="noisy_kappa_train.npy",data=noisy_kappa_train)
# Utility.save_np(data_dir=DATA_DIR, file_name="label_train.npy",data=label_train)
# Utility.save_np(data_dir=DATA_DIR, file_name="noisy_kappa_val.npy",data=noisy_kappa_val)
# Utility.save_np(data_dir=DATA_DIR, file_name="label_val.npy",data=label_val)

In [ ]:
# del noisy_kappa
# gc.collect()

In [ ]:
# # Load the saved split data (if you saved it at DATA_DIR before)

# noisy_kappa_train = Utility.load_np(data_dir=DATA_DIR, file_name="noisy_kappa_train.npy")
# label_train = Utility.load_np(data_dir=DATA_DIR, file_name="label_train.npy")
# noisy_kappa_val = Utility.load_np(data_dir=DATA_DIR, file_name="noisy_kappa_val.npy")
# label_val = Utility.load_np(data_dir=DATA_DIR, file_name="label_val.npy")

# Ntrain = label_train.shape[0]*label_train.shape[1]
# Nval = label_val.shape[0]*label_val.shape[1]

### Reshape data

In [ ]:
# Reshape the data for CNN
X_train = noisy_kappa_train.reshape(Ntrain, *data_obj.shape)
X_val = noisy_kappa_val.reshape(Nval, *data_obj.shape)

# Here, we ignore the nuisance parameters and only keep the 2 cosmological parameters
y_train = label_train.reshape(Ntrain, 5)[:, :2]
y_val = label_val.reshape(Nval, 5)[:, :2]

In [ ]:
print(f'Shape of the split training data = {X_train.shape}')
print(f'Shape of the split validation data = {X_val.shape}')

print(f'Shape of the split training labels = {y_train.shape}')
print(f'Shape of the split validation labels = {y_val.shape}')

### Standartize the data

In [ ]:
del noisy_kappa
del noisy_kappa_train
del noisy_kappa_val
import gc
gc.collect()

In [ ]:
# Compute the means and stds of the training images (for standardizing the data)
means = np.mean(X_train, dtype=np.float32)
stds = np.std(X_train, dtype=np.float32)

# Image standardization
from torchvision import transforms
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[means], std=[stds]),
])
print(f"Image stats (from train set): Mean={means}, Std={stds}")

# Label standardization
label_scaler = StandardScaler()
y_train_scaled = label_scaler.fit_transform(y_train)
y_val_scaled = label_scaler.transform(y_val)
print(f"Label stats (from train set): Mean={label_scaler.mean_}, Std={np.sqrt(label_scaler.var_)}")

# 3 - Visualization

### 2D training maps

survey mask: a binary map that shows which parts of the sky are observed (yellow) and which areas are blocked (purple)

In [ ]:
# mask
Visualization.plot_mask(mask=data_obj.mask)

noiseless training convergence map: The convergence maps show the projected matter density (including dark matter and ordinary matter) in the simulated universe, under the Born approximation. On large scales, we can see the matter forms web-like structures (cosmic web) in the universe. The dense regions in these maps, called dark matter halos, are the sites where galaxies form and reside.

In [ ]:
# noiseless training convergence map
Visualization.plot_noiseless_training_convergence_map(kappa=data_obj.kappa)

noisy training convergence map: We add Gaussian noise to the data. This mimics the observed data. During training the noise can be added on the fly with different realizations.

In [ ]:
# noisy training convergence map
Visualization.plot_noisy_training_convergence_map(kappa=data_obj.kappa,
                                                  mask=data_obj.mask,
                                                  pixelsize_arcmin=data_obj.pixelsize_arcmin,
                                                  ng=data_obj.ng)

### Distribution of physical parameters

Distribution of cosmological parameters $\Omega_m$ and $S_8$. The density increases towards fiducial cosmology. Note that this distribution introduces a prior in the analysis. The test data cosmology follows the same distribution as the training data.

In [ ]:
Visualization.plot_cosmological_parameters_OmegaM_S8(label=data_obj.viz_label)

Distribution of baryonic physics parameters. These are nuisance parameters and should be marginalized in the analysis. They follow a uniform distribution within the prior range $T_{\mathrm{AGN}} \in [7.2, 8.5]$, $f_0 \in [0, 0.0265]$

In [ ]:
Visualization.plot_baryonic_physics_parameters(label=data_obj.viz_label)

Distribution of photometric redshift uncertainty parameters. This is a nuisance parameter and should be marginalized in the analysis. It follows a Gaussian distribution with mean 0 and std 0.022

In [ ]:
Visualization.plot_photometric_redshift_uncertainty_parameters(label=data_obj.viz_label)

# 4 - Training

For each 2D map, the CNN predict its cosmological parameters $(\hat{\Omega}_m, \hat{S}_8)$ and the standard deviations of the joint Gaussian posterior distribution $(\hat{\sigma}_{\Omega_m}, \hat{\sigma}_{S_8})$.

The loss funciton here is a KL divergence objective function defined by
$$
\text{KL Loss}= \frac{1}{N} \sum_i^{N}\left\{\frac{\left(\hat{\Omega}_{m, i}-\Omega_{m, i}^{\text {truth }}\right)^2}{\hat{\sigma}_{\Omega_m, i}^2}+\frac{\left(\hat{S}_{8, i}-S_{8, i}^{\text {truth }}\right)^2}{\hat{\sigma}_{S_8, i}^2}+\log \left(\hat{\sigma}_{\Omega_m, i}^2\right)+\log \left(\hat{\sigma}_{S_8, i}^2\right)\right\}~.
$$

Here are the definitions of Baseline model, CNN1 and CNN2

## Choosing model

Set `USE_PRETRAINED_MODEL = False` if you want to train a new model.\
Set `USE_PRETRAINED_MODEL = True` if you want to load a pretrained model.

In [ ]:
# load_model = "Baseline"
# load_model = "CNN1"
# load_model = "CNN2"
# load_model= "Denario"
# load_model = "CNN3"
load_model = "CNN4"

USE_PRETRAINED_MODEL = False
if USE_PRETRAINED_MODEL:
  PTH_LOAD_PATH = "/content/denario.pth"
  # PTH_LOAD_PATH="/content/"+load_model+".pth" #If we use a pretrained model, set path to the .pth file
else:
  PTH_SAVE_PATH = "/content/"+load_model+".pth"

## Model Definition

### Baseline

In [ ]:
if load_model=="Baseline":
  # Define path for saving the trained model
  MODEL_SAVE_PATH = os.path.join(root_dir, PTH_SAVE_PATH)

  class Simple_CNN(nn.Module):
      def __init__(self, height, width, num_targets):
          super(Simple_CNN, self).__init__()
          # Convolutional layers
          self.conv_stack = nn.Sequential(
              nn.Conv2d(1, 16, kernel_size=5, stride=2, padding=2),
              nn.BatchNorm2d(16),
              nn.ReLU(),
              nn.MaxPool2d(kernel_size=2, stride=2),

              nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1),
              nn.BatchNorm2d(32),
              nn.ReLU(),
              nn.MaxPool2d(kernel_size=2, stride=2),

              nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
              nn.BatchNorm2d(64),
              nn.ReLU(),
              nn.MaxPool2d(kernel_size=2, stride=2),

              nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
              nn.BatchNorm2d(128),
              nn.ReLU(),
              nn.MaxPool2d(kernel_size=2, stride=2)
          )

          self._feature_size = self._get_conv_output_size(height, width)

          # Fully connected layers (regressor head)
          self.fc_stack = nn.Sequential(
              nn.Flatten(),
              nn.Linear(self._feature_size, 512),
              nn.ReLU(),
              nn.Dropout(0.2),
              nn.Linear(512, 128),
              nn.ReLU(),
              nn.Dropout(0.1),
              nn.Linear(128, num_targets)
          )

      def _get_conv_output_size(self, height, width):
          dummy_input = torch.zeros(1, 1, height, width)
          output = self.conv_stack(dummy_input)
          return int(np.prod(output.size()))

      def forward(self, x):
          x = self.conv_stack(x)
          x = self.fc_stack(x)
          means = x[:, :2]
          log_sigmas = x[:, 2:]    # Predict log(σ) to ensure positivity
          sigmas = torch.exp(log_sigmas)
          return means, sigmas     # Note that means and sigmas here have to be rescaled properly due to standardization

  def KL_div_posterior_loss(pred_means, pred_sigmas, truths):
      """
      A KL divergence loss function that directly optimizes the score function

      Inputs:
      - pred_means:   2D tensor (batch_size, 2)
      - pred_sigmas:  2D tensor (batch_size, 2)
      - truths:       2D tensor (batch_size, 2)
      """

      residuals_sq = (pred_means - truths)**2

      loss_terms = residuals_sq / (pred_sigmas**2)
      loss_sum = torch.sum(loss_terms, dim=1)

      log_sigma_terms = torch.sum(torch.log(pred_sigmas**2), dim=1)
      loss = torch.mean(loss_sum + log_sigma_terms)

      return loss

### CNN1

Assymetric convolutions, less parameters (~5M), SiLU instead of ReLU (but still ReLU in dnn part), loss is like score metric.

In [ ]:
if load_model=="CNN1":
  # Define your path for saving the trained model
  MODEL_SAVE_PATH = os.path.join(root_dir, PTH_SAVE_PATH)

  # Enhanced CNN architecture for parameter estimation
  # (still called simple so that the code will still work)
  class Simple_CNN(nn.Module):
      def __init__(self, height, width, num_targets):
          super(Simple_CNN, self).__init__()
          # Convolutional layers
          self.conv_stack = nn.Sequential(
              nn.Conv2d(1, 8, kernel_size=(9,3), stride=(1,1), padding=(4,1)),
              nn.BatchNorm2d(8),
              nn.SiLU(),
              nn.MaxPool2d(kernel_size=3, stride=3),

              nn.Conv2d(8, 16, kernel_size=(7,3), stride=(1,1), padding=(3,1)),
              nn.BatchNorm2d(16),
              nn.SiLU(),
              nn.MaxPool2d(kernel_size=3, stride=3),

              nn.Conv2d(16, 32, kernel_size=(5,3), stride=(1,1), padding=(2,1)),
              nn.BatchNorm2d(32),
              nn.SiLU(),
              nn.MaxPool2d(kernel_size=2, stride=2),

              nn.Conv2d(32, 64, kernel_size=(3,3), stride=(1,1), padding=(1,1)),
              nn.BatchNorm2d(64),
              nn.SiLU(),
              nn.MaxPool2d(kernel_size=2, stride=2)
          )

          self._feature_size = self._get_conv_output_size(height, width)

          # Fully connected layers (regressor head)
          self.fc_stack = nn.Sequential(
              nn.Flatten(),
              nn.Linear(self._feature_size, 512),
              nn.ReLU(),
              # nn.Tanh(), #!!! not pretty sure if this makes any better, may replace with ReLU
              nn.Dropout(0.2),
              nn.Linear(512, 128),
              nn.ReLU(),
              # nn.Tanh(), #!!! same thing
              nn.Dropout(0.1),
              nn.Linear(128, num_targets)
          )

      def _get_conv_output_size(self, height, width):
          dummy_input = torch.zeros(1, 1, height, width)
          output = self.conv_stack(dummy_input)
          return int(np.prod(output.size()))

      def forward(self, x):
          x = self.conv_stack(x)
          x = self.fc_stack(x)
          means = x[:, :2]
          log_sigmas = x[:, 2:]    # Predict log(σ) to ensure positivity
          sigmas = torch.exp(log_sigmas)
          return means, sigmas     # Note that means and sigmas here have to be rescaled properly due to standardization

  def KL_div_posterior_loss(pred_means, pred_sigmas, truths):
      """
      A KL divergence loss function that directly optimizes the score function

      Inputs:
      - pred_means:   2D tensor (batch_size, 2)
      - pred_sigmas:  2D tensor (batch_size, 2)
      - truths:       2D tensor (batch_size, 2)
      """
      # !!! new loss func, more like a scoring function
      lambd = 1000
      residuals_sq = (pred_means - truths)**2
      loss_terms = residuals_sq / (pred_sigmas**2)
      loss_sum = torch.sum(loss_terms, dim=1)
      log_sigma_terms = torch.sum(torch.log(pred_sigmas**2), dim=1)
      exact_term = torch.sum(residuals_sq, dim=1)
      loss = torch.mean(loss_sum + log_sigma_terms + lambd*exact_term)
      return loss

### CNN2

All enhancements of Ours CNN - 1 plus Dual Pooling and Spatial Attention Module.

In [ ]:
if load_model=="CNN2":
  # Define your path for saving the trained model
  MODEL_SAVE_PATH = os.path.join(root_dir, PTH_SAVE_PATH)

  #Dual pooling
  class DualPool2d(nn.Module):
      def __init__(self, kernel_size, stride=None, padding=0):
          super().__init__()
          self.max_pool = nn.MaxPool2d(kernel_size, stride, padding)
          self.avg_pool = nn.AvgPool2d(kernel_size, stride, padding)
      def forward(self, x):
          max_out = self.max_pool(x)
          avg_out = self.avg_pool(x)
          # Concatenate along the channel dimension
          return torch.cat([max_out, avg_out], dim=1)
  #Spatial attention
  class SpatialAttention(nn.Module):
      def __init__(self, kernel_size=7):
          super(SpatialAttention, self).__init__()
          self.conv = nn.Conv2d(2, 1, kernel_size=kernel_size, padding=kernel_size//2, bias=False)
          self.sigmoid = nn.Sigmoid()
      def forward(self, x):
          # Channel pooling: max and avg along channel dimension
          avg_out = torch.mean(x, dim=1, keepdim=True)
          max_out, _ = torch.max(x, dim=1, keepdim=True)
          # Concatenate and apply convolution
          attention = torch.cat([avg_out, max_out], dim=1)
          attention = self.conv(attention)
          attention = self.sigmoid(attention)
          return x * attention
  # Further enhanced CNN architecture
  class Simple_CNN(nn.Module):
      def __init__(self, height, width, num_targets):
          super(Simple_CNN, self).__init__()
          # Convolutional layers
          self.conv_stack = nn.Sequential(
              nn.Conv2d(1, 8, kernel_size=(9,3), stride=(1,1), padding=(4,1)),
              nn.BatchNorm2d(8),
              nn.SiLU(),
              SpatialAttention(kernel_size=3), #!!! spatial attention
              DualPool2d(kernel_size=3, stride=3), #!!! dual pooling

              nn.Conv2d(16, 16, kernel_size=(7,3), stride=(1,1), padding=(3,1)),
              nn.BatchNorm2d(16),
              nn.SiLU(),
              SpatialAttention(kernel_size=3),
              DualPool2d(kernel_size=3, stride=3),

              nn.Conv2d(32, 32, kernel_size=(5,3), stride=(1,1), padding=(2,1)),
              nn.BatchNorm2d(32),
              nn.SiLU(),
              DualPool2d(kernel_size=2, stride=2),

              nn.Conv2d(64, 64, kernel_size=(3,3), stride=(1,1), padding=(1,1)),
              nn.BatchNorm2d(64),
              nn.SiLU(),
              DualPool2d(kernel_size=2, stride=2)
          )

          self._feature_size = self._get_conv_output_size(height, width)

          # Fully connected layers (regressor head)
          self.fc_stack = nn.Sequential(
              nn.Flatten(),
              nn.Linear(self._feature_size, 256),
              nn.ReLU(), #!!! not pretty sure if this makes any better, may replace with ReLU
              nn.Dropout(0.2),
              nn.Linear(256, 128), # !!! also reduced amount of neurons (to decrease amt of parameters)
              nn.ReLU(), #!!! same thing
              nn.Dropout(0.1),
              nn.Linear(128, num_targets)
          )

      def _get_conv_output_size(self, height, width):
          dummy_input = torch.zeros(1, 1, height, width)
          output = self.conv_stack(dummy_input)
          return int(np.prod(output.size()))

      def forward(self, x):
          x = self.conv_stack(x)
          x = self.fc_stack(x)
          means = x[:, :2]
          log_sigmas = x[:, 2:]    # Predict log(σ) to ensure positivity
          sigmas = torch.exp(log_sigmas)
          return means, sigmas     # Note that means and sigmas here have to be rescaled properly due to standardization

  def KL_div_posterior_loss(pred_means, pred_sigmas, truths):
      """
      A KL divergence loss function that directly optimizes the score function

      Inputs:
      - pred_means:   2D tensor (batch_size, 2)
      - pred_sigmas:  2D tensor (batch_size, 2)
      - truths:       2D tensor (batch_size, 2)
      """
      # !!! new loss func, more like a scoring function
      lambd = 1000
      residuals_sq = (pred_means - truths)**2
      loss_terms = residuals_sq / (pred_sigmas**2)
      loss_sum = torch.sum(loss_terms, dim=1)
      log_sigma_terms = torch.sum(torch.log(pred_sigmas**2), dim=1)
      exact_term = torch.sum(residuals_sq, dim=1)
      loss = torch.mean(loss_sum + log_sigma_terms + lambd*exact_term)
      return loss

### CNN3

All enhancements of Ours CNN - 2, but exp applied for sigmas at the output replaced by softplus, stride changed, added one more convolutional layer that reduces amount of parameters.

In [ ]:
if load_model=="CNN3":
  MODEL_SAVE_PATH = os.path.join(root_dir, PTH_SAVE_PATH)
  class DualPool2d(nn.Module):
      def __init__(self, kernel_size, stride=None, padding=0):
          super().__init__()
          self.max_pool = nn.MaxPool2d(kernel_size, stride, padding)
          self.avg_pool = nn.AvgPool2d(kernel_size, stride, padding)
      def forward(self, x):
          max_out = self.max_pool(x)
          avg_out = self.avg_pool(x)
          # Concatenate along the channel dimension
          return torch.cat([max_out, avg_out], dim=1)
  #Spatial attention
  class SpatialAttention(nn.Module):
      def __init__(self, kernel_size=7):
          super(SpatialAttention, self).__init__()
          self.conv = nn.Conv2d(2, 1, kernel_size=kernel_size, padding=kernel_size//2, bias=False)
          self.sigmoid = nn.Sigmoid()
      def forward(self, x):
          # Channel pooling: max and avg along channel dimension
          avg_out = torch.mean(x, dim=1, keepdim=True)
          max_out, _ = torch.max(x, dim=1, keepdim=True)
          # Concatenate and apply convolution
          attention = torch.cat([avg_out, max_out], dim=1)
          attention = self.conv(attention)
          attention = self.sigmoid(attention)
          return x * attention
  # Further enhanced CNN architecture
  class Simple_CNN(nn.Module):
      def __init__(self, height, width, num_targets):
          super(Simple_CNN, self).__init__()
          #Softplus activation
          self.Softplus = torch.nn.Softplus()
          # Convolutional layers
          self.conv_stack = nn.Sequential(
              nn.Conv2d(1, 8, kernel_size=(9,3), stride=(4,1), padding=(4,1)),
              nn.BatchNorm2d(8),
              nn.SiLU(),
              SpatialAttention(kernel_size=3), #!!! spatial attention
              DualPool2d(kernel_size=3, stride=2), #!!! dual pooling

              nn.Conv2d(16, 16, kernel_size=(7,3), stride=(3,1), padding=(3,1)),
              nn.BatchNorm2d(16),
              nn.SiLU(),
              SpatialAttention(kernel_size=3),
              DualPool2d(kernel_size=3, stride=2),

              nn.Conv2d(32, 32, kernel_size=(5,3), stride=(2,1), padding=(2,1)),
              nn.BatchNorm2d(32),
              nn.SiLU(),
              DualPool2d(kernel_size=2, stride=1),

              nn.Conv2d(64, 64, kernel_size=(3,3), stride=(1,1), padding=(1,1)),
              nn.BatchNorm2d(64),
              nn.SiLU(),
              DualPool2d(kernel_size=2, stride=1),

              nn.Conv2d(128, 8, kernel_size=1, stride=(1,1), padding=(1,1)),
              nn.BatchNorm2d(8),
              nn.SiLU()
          )

          self._feature_size = self._get_conv_output_size(height, width)

          # Fully connected layers (regressor head)
          self.fc_stack = nn.Sequential(
              nn.Flatten(),
              nn.Linear(self._feature_size, 256),
              nn.ReLU(), #Tanh showed bad performance here, lets leave ReLU
              nn.Dropout(0.2),
              nn.Linear(256, 128), # !!! also reduced amount of neurons (to decrease amt of parameters)
              nn.ReLU(), #!!! same thing
              nn.Dropout(0.1),
              nn.Linear(128, num_targets)
          )

      def _get_conv_output_size(self, height, width):
          dummy_input = torch.zeros(1, 1, height, width)
          output = self.conv_stack(dummy_input)
          return int(np.prod(output.size()))

      def forward(self, x):
          x = self.conv_stack(x)
          x = self.fc_stack(x)
          means = x[:, :2]
          raw_sigmas = x[:, 2:]
          sigmas = self.Softplus(raw_sigmas) #torch.exp(log_sigmas)
          return means, sigmas     # Note that means and sigmas here have to be rescaled properly due to standardization

  def KL_div_posterior_loss(pred_means, pred_sigmas, truths):
      """
      A KL divergence loss function that directly optimizes the score function

      Inputs:
      - pred_means:   2D tensor (batch_size, 2)
      - pred_sigmas:  2D tensor (batch_size, 2)
      - truths:       2D tensor (batch_size, 2)
      """
      # !!! new loss func, more like a scoring function
      lambd = 1000
      residuals_sq = (pred_means - truths)**2
      loss_terms = residuals_sq / (pred_sigmas**2)
      loss_sum = torch.sum(loss_terms, dim=1)
      log_sigma_terms = torch.sum(torch.log(pred_sigmas**2), dim=1)
      exact_term = torch.sum(residuals_sq, dim=1)
      loss = torch.mean(loss_sum + log_sigma_terms + lambd*exact_term)
      return loss

### CNN4

Literally Ours CNN - 3 but large amount of parameters is reduced by pooling, not convolutions

In [ ]:
if load_model=="CNN4":

  MODEL_SAVE_PATH = os.path.join(root_dir, PTH_SAVE_PATH)
  #Dual pooling
  class DualPool2d(nn.Module):
      def __init__(self, kernel_size, stride=None, padding=0):
          super().__init__()
          self.max_pool = nn.MaxPool2d(kernel_size, stride, padding)
          self.avg_pool = nn.AvgPool2d(kernel_size, stride, padding)
      def forward(self, x):
          max_out = self.max_pool(x)
          avg_out = self.avg_pool(x)
          # Concatenate along the channel dimension
          return torch.cat([max_out, avg_out], dim=1)
      
  #Spatial attention
  class SpatialAttention(nn.Module):
      def __init__(self, kernel_size=7):
          super(SpatialAttention, self).__init__()
          self.conv = nn.Conv2d(2, 1, kernel_size=kernel_size, padding=kernel_size//2, bias=False)
          self.sigmoid = nn.Sigmoid()
      def forward(self, x):
          # Channel pooling: max and avg along channel dimension
          avg_out = torch.mean(x, dim=1, keepdim=True)
          max_out, _ = torch.max(x, dim=1, keepdim=True)
          # Concatenate and apply convolution
          attention = torch.cat([avg_out, max_out], dim=1)
          attention = self.conv(attention)
          attention = self.sigmoid(attention)
          return x * attention
  # Further enhanced CNN architecture
  class Simple_CNN(nn.Module):
      def __init__(self, height, width, num_targets):
          super(Simple_CNN, self).__init__()
          #Softplus activation
          self.Softplus = torch.nn.Softplus()
          # Convolutional layers
          self.conv_stack = nn.Sequential(
              nn.Conv2d(1, 8, kernel_size=(9,3), stride=(4,1), padding=(4,1)),
              nn.BatchNorm2d(8),
              nn.SiLU(),
              SpatialAttention(kernel_size=3), #!!! spatial attention
              DualPool2d(kernel_size=3, stride=2), #!!! dual pooling

              nn.Conv2d(16, 16, kernel_size=(7,3), stride=(3,1), padding=(3,1)),
              nn.BatchNorm2d(16),
              nn.SiLU(),
              SpatialAttention(kernel_size=3),
              DualPool2d(kernel_size=3, stride=2),

              nn.Conv2d(32, 32, kernel_size=(5,3), stride=(2,1), padding=(2,1)),
              nn.BatchNorm2d(32),
              nn.SiLU(),
              DualPool2d(kernel_size=2, stride=1),

              nn.Conv2d(64, 64, kernel_size=(3,3), stride=(1,1), padding=(1,1)),
              nn.BatchNorm2d(64),
              nn.SiLU(),
              DualPool2d(kernel_size=2, stride=1)
          )

          self._feature_size = 6400 #self._get_conv_output_size(height, width)

          # Fully connected layers (regressor head)
          self.fc_stack = nn.Sequential(
              nn.Flatten(),
              nn.Linear(self._feature_size, 256),
              nn.ReLU(), #Tanh showed bad performance here, lets leave ReLU
              nn.Dropout(0.2),
              nn.Linear(256, 128), # !!! also reduced amount of neurons (to decrease amt of parameters)
              nn.ReLU(), #!!! same thing
              nn.Dropout(0.1),
              nn.Linear(128, num_targets)
          )

      def _get_conv_output_size(self, height, width):
          dummy_input = torch.zeros(1, 1, height, width)
          output = self.conv_stack(dummy_input)
          return int(np.prod(output.size()))

      def forward(self, x):
          x = self.conv_stack(x)
          #to dramatically reduce amount of parameters lets use avg and max pooling
          x_avg = torch.nn.functional.adaptive_avg_pool2d(x, (5,5)) # Changed
          x_max = torch.nn.functional.adaptive_max_pool2d(x, (5,5)) # Changed
          x = torch.cat([x_avg, x_max], dim=1)
          x = torch.flatten(x, 1) #new line
          x = self.fc_stack(x)
          means = x[:, :2]
          raw_sigmas = x[:, 2:]
          sigmas = self.Softplus(raw_sigmas) #torch.exp(log_sigmas)
          return means, sigmas     # Note that means and sigmas here have to be rescaled properly due to standardization

  def KL_div_posterior_loss(pred_means, pred_sigmas, truths):
      """
      A KL divergence loss function that directly optimizes the score function

      Inputs:
      - pred_means:   2D tensor (batch_size, 2)
      - pred_sigmas:  2D tensor (batch_size, 2)
      - truths:       2D tensor (batch_size, 2)
      """
      lambd = 1000
      residuals_sq = (pred_means - truths)**2
      loss_terms = residuals_sq / (pred_sigmas**2)
      loss_sum = torch.sum(loss_terms, dim=1)
      log_sigma_terms = torch.sum(torch.log(pred_sigmas**2), dim=1)
      exact_term = torch.sum(residuals_sq, dim=1)
      loss = torch.mean(loss_sum + log_sigma_terms + lambd*exact_term)
      return loss

## Load model and Config

In [ ]:
if USE_PRETRAINED_MODEL:
  MODEL_LOAD_PATH = os.path.join(root_dir, PTH_LOAD_PATH)

class Config:
    IMG_HEIGHT = data_obj.shape[0]
    IMG_WIDTH = data_obj.shape[1]

    # Parameters to predict (Omega_m, S_8, sigma_Omega_m, sigma_S_8)
    NUM_TARGETS = 4

    # Training hyperparameters
    BATCH_SIZE = 64
    EPOCHS = 15
    LEARNING_RATE = 2e-4
    WEIGHT_DECAY = 1e-4   # L2 regularization to prevent overfitting

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

    if USE_PRETRAINED_MODEL:
      MODEL_LOAD_PATH = MODEL_LOAD_PATH
    else :
      MODEL_SAVE_PATH = PTH_SAVE_PATH

## Train and validation

In [ ]:
def train_epoch(model, dataloader, loss_fn, optimizer, device):
    """Trains the model for one epoch."""
    model.train()
    total_loss = 0
    pbar = tqdm(dataloader, total=len(dataloader), desc="Training")
    for X, y in pbar:
        X, y = X.to(device), y.to(device)

        # Forward pass
        pred_means, pred_sigmas = model(X)
        loss = loss_fn(pred_means, pred_sigmas, y)

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)


def validate_epoch(model, dataloader, loss_fn, device):
    """Validates the model on the validation/test set."""
    model.eval()
    total_loss = 0
    pbar = tqdm(dataloader, total=len(dataloader), desc="Validating")
    with torch.no_grad():
        for X, y in pbar:
            X, y = X.to(device), y.to(device)
            pred_means, pred_sigmas = model(X)
            total_loss += loss_fn(pred_means, pred_sigmas, y).item()

    return total_loss / len(dataloader)

In [ ]:
# # Load the configuration
config = Config()
print(f"Using device: {config.DEVICE}")

# # # Create Datasets and DataLoaders
# if not USE_PRETRAINED_MODEL: # We do not need to load train part if we use pretrained model
train_dataset = CosmologyDataset(data=X_train, labels=y_train_scaled, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, shuffle=True)

val_dataset = CosmologyDataset(data=X_val, labels=y_val_scaled, transform=transform)
val_loader = DataLoader(val_dataset, batch_size=config.BATCH_SIZE, shuffle=False)

In [ ]:
if load_model=="Denario":
  from improved_training_pipeline import EnhancedCNN #We have to add improved_training_pipeline.py to /content in colab

In [ ]:
if not USE_PRETRAINED_MODEL:
    # Initialize the baseline CNN model
    model = Simple_CNN(config.IMG_HEIGHT, config.IMG_WIDTH, config.NUM_TARGETS).to(config.DEVICE)
    # Train the model
    loss_fn = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=config.LEARNING_RATE, weight_decay=config.WEIGHT_DECAY)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)
    # Training Loop
    best_val_loss = float('inf')
    start_time = time.time()
    for epoch in range(config.EPOCHS): #one model is continuosly trained here
        train_loss = train_epoch(model, train_loader, KL_div_posterior_loss, optimizer, config.DEVICE)
        val_loss = validate_epoch(model, val_loader, KL_div_posterior_loss, config.DEVICE)

        scheduler.step(val_loss)
        print(f"Epoch {epoch+1}/{config.EPOCHS} | Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f}")

        # Save the best model based on validation loss
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), config.MODEL_SAVE_PATH)
            print(f"  -> New best model saved to {config.MODEL_SAVE_PATH}")

    end_time = time.time()
    print(f"\nTraining finished in {(end_time - start_time)/60:.2f} minutes.")

    model.load_state_dict(torch.load(config.MODEL_SAVE_PATH, weights_only=True)) # Directly load the best model

else: # Here EnhancedCNN is the model from Denario
    if load_model=="Denario":
        model = EnhancedCNN()
    else:
      model = Simple_CNN(config.IMG_HEIGHT, config.IMG_WIDTH, config.NUM_TARGETS).to(config.DEVICE)
    if os.path.exists(config.MODEL_LOAD_PATH):
        model.load_state_dict(torch.load(config.MODEL_LOAD_PATH, weights_only=True, map_location=torch.device('cpu')))
        model = model.to(config.DEVICE)
    else:
        warning_msg = f"The path of pretrained model doesn't exist"
        warnings.warn(warning_msg)

# 5 - Inference on the validation set

In [ ]:
def predict_dataset(model, loader, device, scaler):
    model.eval()
    means_list, sigmas_list = [], []
    pbar = tqdm(loader, total=len(loader), desc="Inference")
    with torch.no_grad():
        for X, _ in pbar:
            X = X.to(device)
            means, sigmas = model(X)
            means_list.append(means.cpu().numpy())
            sigmas_list.append(sigmas.cpu().numpy())

    mean = np.concatenate(means_list, axis=0)
    sigma = np.concatenate(sigmas_list, axis=0)

    mean = scaler.inverse_transform(mean)
    sigma = sigma * np.sqrt(scaler.var_)

    return mean, sigma

In [ ]:
def plot_posterior_vs_truth(y_true, mean, sigma, idx, name, ylim=None):
    """
    y_true : array (N, D)
    mean   : array (N, D)
    sigma  : array (N, D)
    idx    : index of the parameter (0 = Omega_m, 1 = S_8)
    name   : parameter name for the title
    ylim   : tuple for y-axis limits
    """

    plt.figure(figsize=(6,5))

    # Scatter + error bars
    plt.errorbar(
        y_true[:, idx],
        mean[:, idx],
        yerr=sigma[:, idx],
        fmt='o', markersize=4,
        capsize=3, capthick=1,
        ecolor='grey', elinewidth=1
    )

    # Perfect prediction line
    sorted_truth = np.sort(y_true[:, idx])
    plt.plot(sorted_truth, sorted_truth,
             color='grey', linestyle='dashed')

    # Axis labels & title
    plt.xlabel('Ground Truth')
    plt.ylabel('Prediction')
    plt.title(name)

    # Optional y-limits
    if ylim is not None:
        plt.ylim(*ylim)

    # Consistent x-limits
    plt.xlim(np.min(y_true[:, idx]), np.max(y_true[:, idx]))

    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()




In [ ]:
mean_val, errorbar_val = predict_dataset(model, val_loader, config.DEVICE, label_scaler)
mean_train, errorbar_train = predict_dataset(model, train_loader, config.DEVICE, label_scaler)

In [ ]:
## Include the prior that the cosmological parameters are not negative
negative_mask = mean_train - errorbar_train < 0
errorbar_train[negative_mask] = mean_train[negative_mask]

In [ ]:
## Include the prior that the cosmological parameters are not negative
negative_mask = mean_val - errorbar_val < 0
errorbar_val[negative_mask] = mean_val[negative_mask]

In [ ]:
plot_posterior_vs_truth(
    y_true=y_train, mean=mean_train, sigma=errorbar_train,
    idx=0, name=r'$\Omega_m$',
    ylim=(0, 0.7)
)

plot_posterior_vs_truth(
    y_true=y_train, mean=mean_train, sigma=errorbar_train,
    idx=1, name=r'$S_8$',
    ylim=(0.65, 1)
)

In [ ]:
plot_posterior_vs_truth(
    y_true=y_val, mean=mean_val, sigma=errorbar_val,
    idx=0, name=r'$\Omega_m$',
    ylim=(0, 0.7)
)

plot_posterior_vs_truth(
    y_true=y_val, mean=mean_val, sigma=errorbar_val,
    idx=1, name=r'$S_8$',
    ylim=(0.65, 1)
)

In [ ]:
training_score = Score._score_phase1(true_cosmo=y_train, infer_cosmo=mean_train, errorbar=errorbar_train)
print("Training:")
print('averaged score:', np.mean(training_score))
print('averaged error bar:', np.mean(errorbar_train, 0))

In [ ]:
print("Validation:")
validation_score = Score._score_phase1(true_cosmo=y_val, infer_cosmo=mean_val, errorbar=errorbar_val)
print('averaged score:', np.mean(validation_score))
print('averaged error bar:', np.mean(errorbar_val, 0))

Denario :
averaged score: 8.387599703672834
averaged error bar: [0.04537246 0.0302369 ]

Baseline :  

First time : averaged score: 7.735698835045951
averaged error bar: [0.04517587 0.03164865]

Second time :
averaged score: 7.781933995160765
averaged error bar: [0.03958124 0.02604696]